# 33 - Oversample: fetch up to 2000 companies per query from the v2 API

Fetches up to 2000 results per query (all 101 queries used across the thesis) from the Istari v2 API, to build an extra, larger companies dataset on top of the existing 1000-per-query ground truth pool.

Saved after every single page (not just every query), to JSON, so an interruption (quota exhaustion, rate limit, crash) never loses more than one in-flight page. Rerunning this notebook resumes exactly where it left off, no query or page is ever re-fetched once saved.

Sizing is budget-aware: it checks the live remaining monthly quota first and never asks for more than that, so it degrades gracefully instead of erroring out if quota is tight this month. If quota runs out partway through, whatever was fetched so far is already safely on disk, just rerun this notebook next month (or once quota is topped up) to continue.

In [1]:
import os, json, time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

ISTARI_API_KEY = os.getenv("API_KEY")
V2_BASE_URL    = "https://api.istari.ai/v2/search"
RESULT_DIR     = Path("result/33_api_v2_oversample")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH     = RESULT_DIR / "oversample_cache.json"

TARGET_SIZE_PER_QUERY = 2000
V2_MAX_PAGE_SIZE       = 500
V2_COLUMNS             = ["domain", "name", "country", "summary", "organization_type", "organization_size", "nace_code", "summary_keywords"]
V2_RATE_LIMIT_RETRIES  = 5
V2_RATE_LIMIT_WAIT     = 10
V2_PAGE_SLEEP          = 1.5
V2_QUERY_SLEEP         = 3.0

print(f"API key set: {'yes' if ISTARI_API_KEY else 'NO -- check API_KEY in .env'}")
print(f"Cache file : {CACHE_PATH}")

API key set: yes
Cache file : result/33_api_v2_oversample/oversample_cache.json


In [2]:
# Query list: reuse the exact same 101 (query_id, query) pairs already used across the thesis, so this dataset lines up with everything else.
corpus = pd.read_csv("dataset/company_corpus.csv")
queries_df = corpus[["query_id", "query"]].drop_duplicates().sort_values("query_id").reset_index(drop=True)
QUERIES = list(zip(queries_df["query_id"].astype(int), queries_df["query"]))
print(f"Loaded {len(QUERIES)} queries")

Loaded 101 queries


In [3]:
# Live quota check -- authoritative, read from response headers, not the dashboard UI.
r = requests.post(
    V2_BASE_URL,
    headers={"Accept": "application/json", "x-api-key": ISTARI_API_KEY, "Content-Type": "application/json"},
    json={"describe": QUERIES[0][1], "keywords": {"must_all": [], "must_any": [], "must_not": []},
          "filters": {"country": [], "state": [], "region": [], "organization_type": [],
                      "organization_size": [], "nace_code": []},
          "excludes": [], "columns": ["domain", "name"], "size": 3},
    timeout=15,
)
if r.status_code != 200:
    raise SystemExit(f"v2 API error {r.status_code}: {r.text[:200]}")

remaining_results_hdr = r.headers.get("X-RateLimit-Results-Remaining")
remaining_requests_hdr = r.headers.get("X-RateLimit-Requests-Remaining")
UNLIMITED_FALLBACK = 10_000_000
REMAINING_RESULTS  = int(remaining_results_hdr) if remaining_results_hdr is not None else UNLIMITED_FALLBACK
REMAINING_REQUESTS = int(remaining_requests_hdr) if remaining_requests_hdr is not None else UNLIMITED_FALLBACK
print(f"Live quota remaining -- results: {REMAINING_RESULTS}, requests: {REMAINING_REQUESTS}")
print(f"Full target would need up to {TARGET_SIZE_PER_QUERY * len(QUERIES):,} results across {len(QUERIES)} queries.")
print("This run will fetch as much of that as the current quota allows, then stop cleanly -- rerun later to continue.")

Live quota remaining -- results: 2762, requests: 4310
Full target would need up to 202,000 results across 101 queries.
This run will fetch as much of that as the current quota allows, then stop cleanly -- rerun later to continue.


In [4]:
def load_cache():
    if CACHE_PATH.exists():
        with open(CACHE_PATH, "r") as f:
            return json.load(f)
    return {}


def save_cache(cache):
    tmp_path = CACHE_PATH.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(cache, f, indent=2, default=str)
    tmp_path.replace(CACHE_PATH)  # atomic on the same filesystem -- avoids a half-written cache file if interrupted mid-save


cache = load_cache()
print(f"Loaded cache: {len(cache)} queries with existing progress")

Loaded cache: 0 queries with existing progress


In [5]:
def fetch_page(query, size, search_after=None):
    """One page of the v2 API. Returns (data, next_cursor, remaining_results) on success.
    Raises SystemExit on genuine monthly quota exhaustion. Returns (None, None, remaining) on a transient error after retries are exhausted."""
    payload = {"describe": query, "keywords": {"must_all": [], "must_any": [], "must_not": []},
               "filters": {"country": [], "state": [], "region": [], "organization_type": [],
                           "organization_size": [], "nace_code": []},
               "excludes": [], "columns": V2_COLUMNS, "size": size}
    if search_after is not None:
        payload["search_after"] = search_after

    for attempt in range(V2_RATE_LIMIT_RETRIES):
        resp = requests.post(
            V2_BASE_URL,
            headers={"Accept": "application/json", "x-api-key": ISTARI_API_KEY, "Content-Type": "application/json"},
            json=payload, timeout=30,
        )
        remaining_hdr = resp.headers.get("X-RateLimit-Results-Remaining")
        remaining = int(remaining_hdr) if remaining_hdr is not None else None

        if resp.status_code == 200:
            body = resp.json()
            return body.get("data", []), body.get("metadata", {}).get("search_after"), remaining

        if resp.status_code == 429:
            body_text = resp.text[:200]
            quota_exhausted = (remaining is not None and remaining <= 0) or "quota" in body_text.lower()
            if quota_exhausted:
                raise SystemExit(
                    f'[QuotaExhausted] on "{query}" (results remaining: {remaining}). '
                    f"Everything fetched so far is already saved in {CACHE_PATH}. "
                    f"Rerun this notebook once quota resets to continue exactly where this stopped."
                )
            wait = V2_RATE_LIMIT_WAIT * (attempt + 1)
            print(f'    [RateLimit] transient 429 on "{query}" ({remaining} results still remaining) -- waiting {wait}s (attempt {attempt+1}/{V2_RATE_LIMIT_RETRIES})')
            time.sleep(wait)
            continue

        print(f'    [Error] {resp.status_code} on "{query}": {resp.text[:150]}')
        return None, None, remaining

    print(f'    [GaveUp] on "{query}" after {V2_RATE_LIMIT_RETRIES} rate-limit retries')
    return None, None, None

In [6]:
print(f"Fetching up to {TARGET_SIZE_PER_QUERY} companies per query for {len(QUERIES)} queries...")
print("-" * 60)

for query_id, query in QUERIES:
    key = str(query_id)
    entry = cache.get(key, {"query": query, "results": [], "search_after": None, "complete": False})
    cache[key] = entry

    if entry["complete"]:
        continue

    seen_domains = {row["domain"] for row in entry["results"]}

    while len(entry["results"]) < TARGET_SIZE_PER_QUERY:
        page_size = min(V2_MAX_PAGE_SIZE, TARGET_SIZE_PER_QUERY - len(entry["results"]))
        data, next_cursor, remaining = fetch_page(query, page_size, entry["search_after"])

        if data is None:
            print(f'  [Skip] "{query}" -- transient failure, will retry next run (progress so far already saved)')
            break

        new_rows = [row for row in data if row["domain"] not in seen_domains]
        seen_domains.update(row["domain"] for row in new_rows)
        for rank, row in enumerate(new_rows, start=len(entry["results"]) + 1):
            row["rank"] = rank
        entry["results"].extend(new_rows)
        entry["search_after"] = next_cursor

        if not data or next_cursor is None:
            entry["complete"] = True  # corpus has fewer distinct matches than the target, nothing more to fetch

        # Save after EVERY page, not just every query -- this is the actual loss-prevention step.
        save_cache(cache)

        if entry["complete"] or len(entry["results"]) >= TARGET_SIZE_PER_QUERY:
            break
        time.sleep(V2_PAGE_SLEEP)

    if len(entry["results"]) >= TARGET_SIZE_PER_QUERY:
        entry["complete"] = True
        save_cache(cache)

    done_count = sum(1 for e in cache.values() if e["complete"])
    print(f'  [{done_count}/{len(QUERIES)} queries complete]  "{query}" -> {len(entry["results"])} companies')
    time.sleep(V2_QUERY_SLEEP)

print("-" * 60)
print("Done for this run (or stopped early on quota exhaustion above -- either way, nothing fetched has been lost).")

Fetching up to 2000 companies per query for 101 queries...
------------------------------------------------------------
  [1/101 queries complete]  "software companies" -> 2000 companies


SystemExit: [QuotaExhausted] on "healthcare providers" (results remaining: 239). Everything fetched so far is already saved in result/33_api_v2_oversample/oversample_cache.json. Rerun this notebook once quota resets to continue exactly where this stopped.

/home/ma/ma_ma/ma_mpandya/Thesis/thesis/lib64/python3.12/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Consolidate into one dataset

Safe to run any time, even mid-run on a partially-fetched cache, just reads whatever is currently saved.

In [7]:
cache = load_cache()
rows = []
for key, entry in cache.items():
    for row in entry["results"]:
        rows.append({**row, "query_id": int(key), "query": entry["query"]})

combined_df = pd.DataFrame(rows)
combined_path = RESULT_DIR / "oversample_companies.json"
combined_df.to_json(combined_path, orient="records", indent=2)

n_complete = sum(1 for e in cache.values() if e["complete"])
print(f"Queries complete: {n_complete}/{len(QUERIES)}")
print(f"Total companies fetched so far: {len(combined_df)}")
print(f"Saved combined dataset -> {combined_path}")

Queries complete: 1/101
Total companies fetched so far: 2500
Saved combined dataset -> result/33_api_v2_oversample/oversample_companies.json
